# MS_C2 — Multi-Station CNN-LSTM (Perfect Forecast Full (HYSPLIT + MET at t+h))

Trains one Bidirectional CNN-LSTM (MIMO, 24 horizons) per station in `STATIONS_TO_RUN`.
Uses `weather_mode="perfect_forecast_full"` (MET_COLS + HYSPLIT shifted to t+24 in each sequence window).

**Checkpoint:** skips a station if `outputs/{station}/results/C2_metrics.csv` already exists.
A kernel restart resumes from the last incomplete station.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd

import src.config as cfg
import src.data_loader as dl
import src.feature_engineering as fe

from src.config import ALL_STATIONS, HORIZONS, SEQ_LEN_LSTM, RANDOM_SEED, get_station_paths
from src.feature_engineering import build_sequence_dataset, WEATHER_MODE_PERFECT_FULL
from src.models.hybrid_lstm import train_cnn_lstm, predict_cnn_lstm
from src.evaluation import compute_metrics
from src.utils import ensure_dirs, set_seed

set_seed(RANDOM_SEED)

WEATHER_MODE = WEATHER_MODE_PERFECT_FULL
# Override to run a subset, e.g. STATIONS_TO_RUN = ["MzWarChrosci"]
STATIONS_TO_RUN = ALL_STATIONS

print(f'Stations: {STATIONS_TO_RUN}')
print(f'Weather mode: {WEATHER_MODE}')

Stations: ['MzWarChrosci', 'MzOtwoBrzozo', 'MzWarWokalna', 'MzWarAlNiepo', 'MzLegZegrzyn', 'MzPiasPulask', 'MzWarBajkowa']
Weather mode: perfect_forecast_full


In [2]:
wall_start = time.time()
n = len(STATIONS_TO_RUN)

for i, station in enumerate(STATIONS_TO_RUN, 1):
    paths = get_station_paths(station)
    checkpoint = paths['results'] / 'C2_metrics.csv'

    if checkpoint.exists():
        print(f'[{i}/{n}] {station} — already done, skipping')
        continue

    print(f'\n[{i}/{n}] {station} — starting ...')
    t_station = time.time()

    ensure_dirs(paths['models'], paths['figures'], paths['results'])

    # Monkeypatch TARGET so src functions operate on the current station
    cfg.TARGET = station
    dl.TARGET = station
    fe.TARGET = station

    df = dl.load_data()
    train_df, test_df = dl.train_test_split(df)

    # ── Build sequence datasets ───────────────────────────────────────
    X_train_seq, y_train_seq, feature_names, scaler_X = build_sequence_dataset(
        train_df, seq_len=SEQ_LEN_LSTM, horizons=HORIZONS,
        fit_scaler=True, weather_mode=WEATHER_MODE
    )
    X_test_seq, y_test_seq, _, _ = build_sequence_dataset(
        test_df, seq_len=SEQ_LEN_LSTM, horizons=HORIZONS,
        scaler_X=scaler_X, fit_scaler=False, weather_mode=WEATHER_MODE
    )

    # ── Validation split: fixed mid-training block (2022) ────────────
    # Using a mid-block rather than the tail avoids tuning early stopping
    # toward the regime immediately before the 2024 test set.
    # train_df covers 2019–2023; 2022 starts at roughly index 26280 (3*8760).
    val_start = int(3 * 8760)  # approx start of 2022 in hourly training data
    val_end   = val_start + int(0.1 * len(X_train_seq))
    val_end   = min(val_end, len(X_train_seq) - 1)
    X_tr  = np.concatenate([X_train_seq[:val_start], X_train_seq[val_end:]], axis=0)
    y_tr  = np.concatenate([y_train_seq[:val_start], y_train_seq[val_end:]], axis=0)
    X_val = X_train_seq[val_start:val_end]
    y_val = y_train_seq[val_start:val_end]

    print(f'  Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test_seq.shape}')

    # ── Train CNN-LSTM ────────────────────────────────────────────────
    model_path = paths['models'] / 'cnn_lstm_pfx_model.pt'
    model, history = train_cnn_lstm(
        X_tr, y_tr, X_val, y_val,
        epochs=100, batch_size=64, patience=15, lr=1e-3,
        save_path=model_path
    )

    # ── Evaluate on test set ─────────────────────────────────────────
    cnn_preds = predict_cnn_lstm(model, X_test_seq)
    y_test_true = np.expm1(y_test_seq)

    rows = []
    for h_idx, h in enumerate(HORIZONS):
        m = compute_metrics(y_test_true[:, h_idx], cnn_preds[:, h_idx])
        rows.append({'Model': 'C2_CNN_LSTM_pfxf', 'Station': station, 'Horizon': h, **m})

    # Save metrics immediately so checkpoint is valid on next restart
    pd.DataFrame(rows).to_csv(checkpoint, index=False)

    elapsed_min = (time.time() - t_station) / 60
    total_elapsed_min = (time.time() - wall_start) / 60
    avg_per_station = total_elapsed_min / i
    remaining_min = avg_per_station * (n - i)
    print(f'[{i}/{n}] {station} — done | elapsed {elapsed_min:.1f}min | est. remaining {remaining_min:.1f}min')

# Restore original TARGET
cfg.TARGET = 'MzWarChrosci'
dl.TARGET = 'MzWarChrosci'
fe.TARGET = 'MzWarChrosci'

print(f'\nAll stations complete in {(time.time() - wall_start) / 60:.1f}min total.')

15:53:14 | src.data_loader | INFO | Loading data from D:\MOJE\PROJEKTY\warsaw_aq_forecast\data\raw\FINAL_merged_PM25_1g_all_seasons.csv



[1/7] MzWarChrosci — starting ...


15:53:14 | src.data_loader | INFO | Missing values per column:
MzOtwoBrzozo    1084
MzWarWokalna    2028
MzWarAlNiepo     575
MzLegZegrzyn    1511
MzPiasPulask    1002
MzWarChrosci     416
MzWarBajkowa     636
dir_48            48
dir_24            48
15:53:14 | src.data_loader | INFO | Loaded 52608 rows, 27 columns, range 2019-01-01 00:00:00 → 2024-12-31 23:00:00
15:53:14 | src.data_loader | INFO | Train: 43824 rows (2019-01-01 00:00:00 → 2023-12-31 23:00:00)
15:53:14 | src.data_loader | INFO | Test : 8784 rows (2024-01-01 00:00:00 → 2024-12-31 23:00:00)
15:53:14 | src.feature_engineering | INFO | build_sequence_dataset | seq_len=48 | weather_mode=perfect_forecast_full
15:53:14 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -24 h
15:53:14 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
15:53:14 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns

  Train: (34092, 48, 45) | Val: (3787, 48, 45) | Test: (7709, 48, 45)


15:53:24 | src.models.hybrid_lstm | INFO | CNN-LSTM | seq_len=48  n_features=45  n_horizons=24  device=cpu
15:53:24 | src.models.hybrid_lstm | INFO | Params: epochs=100  batch=64  patience=15  lr=0.00100
15:53:24 | src.models.hybrid_lstm | INFO | Training samples=34092  val samples=3787
15:53:24 | src.models.hybrid_lstm | INFO | -----------------------------------------------------------------
15:54:20 | src.models.hybrid_lstm | INFO | Epoch   1/100  train=0.16365  val=0.07641  best=0.07641  lr=1.00e-03 *
15:55:30 | src.models.hybrid_lstm | INFO | Epoch   2/100  train=0.05888  val=0.05700  best=0.05700  lr=1.00e-03 *
15:56:33 | src.models.hybrid_lstm | INFO | Epoch   3/100  train=0.04342  val=0.06195  best=0.05700  lr=1.00e-03  (no improve 1/15)
15:57:31 | src.models.hybrid_lstm | INFO | Epoch   4/100  train=0.03548  val=0.05869  best=0.05700  lr=1.00e-03  (no improve 2/15)
15:58:28 | src.models.hybrid_lstm | INFO | Epoch   5/100  train=0.03113  val=0.06550  best=0.05700  lr=1.00e-03  

...

[6/7] MzPiasPulask — done | elapsed 15.0min | est. remaining 17.0min

[7/7] MzWarBajkowa — starting ...


17:35:12 | src.data_loader | INFO | Missing values per column:
MzOtwoBrzozo    1084
MzWarWokalna    2028
MzWarAlNiepo     575
MzLegZegrzyn    1511
MzPiasPulask    1002
MzWarChrosci     416
MzWarBajkowa     636
dir_48            48
dir_24            48
17:35:12 | src.data_loader | INFO | Loaded 52608 rows, 27 columns, range 2019-01-01 00:00:00 → 2024-12-31 23:00:00
17:35:12 | src.data_loader | INFO | Train: 43824 rows (2019-01-01 00:00:00 → 2023-12-31 23:00:00)
17:35:12 | src.data_loader | INFO | Test : 8784 rows (2024-01-01 00:00:00 → 2024-12-31 23:00:00)
17:35:12 | src.feature_engineering | INFO | build_sequence_dataset | seq_len=48 | weather_mode=perfect_forecast_full
17:35:13 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -24 h
17:35:13 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
17:35:13 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns

  Train: (34424, 48, 46) | Val: (3824, 48, 46) | Test: (7786, 48, 46)


17:35:58 | src.models.hybrid_lstm | INFO | Epoch   1/100  train=0.18588  val=0.06770  best=0.06770  lr=1.00e-03 *
17:36:38 | src.models.hybrid_lstm | INFO | Epoch   2/100  train=0.06959  val=0.07177  best=0.06770  lr=1.00e-03  (no improve 1/15)
17:37:17 | src.models.hybrid_lstm | INFO | Epoch   3/100  train=0.04961  val=0.05467  best=0.05467  lr=1.00e-03 *
17:37:53 | src.models.hybrid_lstm | INFO | Epoch   4/100  train=0.04050  val=0.06540  best=0.05467  lr=1.00e-03  (no improve 1/15)
17:38:37 | src.models.hybrid_lstm | INFO | Epoch   5/100  train=0.03568  val=0.07984  best=0.05467  lr=1.00e-03  (no improve 2/15)
17:39:14 | src.models.hybrid_lstm | INFO | Epoch   6/100  train=0.03186  val=0.07154  best=0.05467  lr=1.00e-03  (no improve 3/15)
17:39:47 | src.models.hybrid_lstm | INFO | Epoch   7/100  train=0.02912  val=0.07561  best=0.05467  lr=1.00e-03  (no improve 4/15)
17:40:23 | src.models.hybrid_lstm | INFO | Epoch   8/100  train=0.02672  val=0.08687  best=0.05467  lr=1.00e-03  (no 

[7/7] MzWarBajkowa — done | elapsed 11.4min | est. remaining 0.0min

All stations complete in 113.4min total.
